# 01. Neural Network Regression with TensorFlow

There are many definitions for a regression problem but in our case, we're going to simplify it to be: **predicting a number**.

**Examples:**
- Predict the selling price of houses given information about them
- Predict the coordinates of a bounding box of an item in an image
- Predict the cost of medical insurance for an individual

## What we're going to cover

- Architecture of a regression model
- Input shapes and output shapes
- Creating custom data to view and fit
- Steps in modelling (Creating, Compiling, Fitting)
- Evaluating and improving a model
- Comparing experiments
- Saving and loading models

## Typical Architecture of a Regression Neural Network

| **Hyperparameter** | **Typical value** |
| --- | --- |
| Input layer shape | Same shape as number of features |
| Hidden layer(s) | Problem specific, minimum = 1 |
| Neurons per hidden layer | Generally 10 to 100 |
| Output layer shape | Same shape as desired prediction shape |
| Hidden activation | ReLU (rectified linear unit) |
| Output activation | None, ReLU, logistic/tanh |
| Loss function | MSE or MAE/Huber |
| Optimizer | SGD, Adam |

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from keras.metrics import MeanAbsoluteError, MeanSquaredError
import datetime

print(f"TensorFlow version: {tf.__version__}")
print(f"Notebook last run: {datetime.datetime.now()}")

## Creating Data to View and Fit

Since we're working on a **regression problem** (predicting a number) let's create some linear data (a straight line) to model.

In [ ]:
# Create features
X = np.array([-7.0, -4.0, -1.0, 2.0, 5.0, 8.0, 11.0, 14.0])

# Create labels (y = X + 10)
y = np.array([3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0])

# Visualize it
plt.figure(figsize=(10, 7))
plt.scatter(X, y)
plt.xlabel("X (Features)")
plt.ylabel("y (Labels)")
plt.title("Simple Linear Data")
plt.show()

## Input and Output Shapes

One of the most important concepts when working with neural networks are the **input and output shapes**.

- The **input shape** is the shape of your data that goes into the model.
- The **output shape** is the shape of your data you want to come out of your model.

In [ ]:
# Example input and output shapes of a regression model
house_info = tf.constant(["bedroom", "bathroom", "garage"])
house_price = tf.constant([939700])

print(f"House info (features): {house_info}")
print(f"House info shape: {house_info.shape}")  # 3 features
print(f"\nHouse price (label): {house_price}")
print(f"House price shape: {house_price.shape}")  # 1 output

In [ ]:
# Create features (using tensors)
X = tf.constant([-7.0, -4.0, -1.0, 2.0, 5.0, 8.0, 11.0, 14.0])
y = tf.constant([3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0])

# Take a single example
input_shape = X[0].shape 
output_shape = y[0].shape

print(f"Single X value: {X[0]}")
print(f"Single y value: {y[0]}")
print(f"Input shape: {input_shape}")  # These are scalars (no shape)
print(f"Output shape: {output_shape}")

## Steps in Modelling with TensorFlow

In TensorFlow, there are typically 3 fundamental steps to creating and training a model:

1. **Creating a model** - piece together the layers of a neural network
2. **Compiling a model** - defining loss function, optimizer, and metrics
3. **Fitting a model** - letting the model try to find patterns in the data

In [ ]:
# Set random seed for reproducibility
tf.random.set_seed(42)

# 1. Create a model using the Sequential API
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1)
])

# 2. Compile the model
model.compile(
    loss=tf.keras.losses.mae,  # Mean Absolute Error
    optimizer=tf.keras.optimizers.SGD(),  # Stochastic Gradient Descent
    metrics=["mae"]
)

# 3. Fit the model (TensorFlow 2.7+ requires expanding dimensions)
model.fit(tf.expand_dims(X, axis=-1), y, epochs=5)

In [ ]:
# Make a prediction with the model
# For X=17.0, expected y = 17 + 10 = 27
x_input = np.array([[17.0]])
prediction = model.predict(x_input)
print(f"Prediction for X=17.0: {prediction[0][0]:.2f}")
print(f"Expected: 27.0")

## Improving a Model

Ways to improve a model:
1. **Creating a model** - add more layers, increase hidden units, change activation functions
2. **Compiling a model** - change optimizer or learning rate
3. **Fitting a model** - train for more epochs or use more data

Let's train for longer (100 epochs instead of 5):

In [ ]:
# Set random seed
tf.random.set_seed(42)

# Create a model (same as before)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1)
])

# Compile model
model.compile(
    loss=tf.keras.losses.mae,
    optimizer=tf.keras.optimizers.SGD(),
    metrics=["mae"]
)

# Fit model for longer (100 epochs)
model.fit(tf.expand_dims(X, axis=-1), y, epochs=100, verbose=0)

# Try prediction again
prediction = model.predict(np.array([[17.0]]), verbose=0)
print(f"Prediction for X=17.0: {prediction[0][0]:.2f}")
print(f"Expected: 27.0")
print(f"Much better after training longer!")

## Making a Bigger Dataset & Train/Test Split

Let's create a bigger dataset and properly split it into training and test sets.

- **Training set** - the model learns from this data (70-80%)
- **Validation set** - the model gets tuned on this data (10-15%)
- **Test set** - the model gets evaluated on this data (10-15%)

In [ ]:
# Make a bigger dataset
X = np.arange(-100, 100, 4)
y = X + 10  # y = X + 10 relationship

print(f"Number of samples: {len(X)}")
print(f"X: {X[:5]}... to {X[-5:]}")
print(f"y: {y[:5]}... to {y[-5:]}")

In [ ]:
# Split data into train and test sets
X_train = X[:40]  # first 40 examples (80% of data)
y_train = y[:40]

X_test = X[40:]   # last 10 examples (20% of data)
y_test = y[40:]

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Visualize the data
plt.figure(figsize=(10, 7))
plt.scatter(X_train, y_train, c='b', label='Training data')
plt.scatter(X_test, y_test, c='g', label='Testing data')
plt.xlabel("X")
plt.ylabel("y")
plt.title("Training vs Test Data")
plt.legend()
plt.show()

## Visualizing the Model

Let's build a model with a defined input shape so we can view its summary.

In [ ]:
# Set random seed
tf.random.set_seed(42)

# Create a model with defined input shape
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=[1])  # define input_shape
])

# Compile the model
model.compile(
    loss=tf.keras.losses.mae,
    optimizer=tf.keras.optimizers.SGD(),
    metrics=["mae"]
)

# View model summary
model.summary()

In [ ]:
# Fit the model to the training data
model.fit(X_train, y_train, epochs=100, verbose=0)
print("Model trained for 100 epochs!")

## Visualizing Predictions

Let's make predictions and compare them to the ground truth.

In [ ]:
# Define a function to plot predictions
def plot_predictions(train_data=X_train, 
                     train_labels=y_train, 
                     test_data=X_test, 
                     test_labels=y_test, 
                     predictions=None):
    """
    Plots training data, test data and compares predictions.
    """
    plt.figure(figsize=(10, 7))
    plt.scatter(train_data, train_labels, c="b", label="Training data")
    plt.scatter(test_data, test_labels, c="g", label="Testing data")
    if predictions is not None:
        plt.scatter(test_data, predictions, c="r", label="Predictions")
    plt.xlabel("X")
    plt.ylabel("y")
    plt.title("Predictions vs Ground Truth")
    plt.legend()
    plt.show()

In [ ]:
# Make predictions
y_preds = model.predict(X_test)

# Plot predictions
plot_predictions(predictions=y_preds)

## Evaluating Predictions

Two main metrics for regression:
- **MAE (Mean Absolute Error)** - mean difference between predictions
- **MSE (Mean Squared Error)** - squared mean difference (penalizes larger errors more)

The **lower** each value, the **better**.

In [ ]:
# Evaluate the model on the test set
loss, mae = model.evaluate(X_test, y_test)
print(f"Loss: {loss:.4f}")
print(f"MAE: {mae:.4f}")

In [ ]:
# Calculate MAE and MSE manually
mae_metric = MeanAbsoluteError()
mse_metric = MeanSquaredError()

mae_value = mae_metric(y_test, y_preds.squeeze()).numpy()
mse_value = mse_metric(y_test, y_preds.squeeze()).numpy()

print(f"MAE: {mae_value:.4f}")
print(f"MSE: {mse_value:.4f}")

## Running Experiments to Improve a Model

Let's compare 3 different models:
1. **model_1** - 1 layer, 100 epochs
2. **model_2** - 2 layers, 100 epochs
3. **model_3** - 2 layers, 500 epochs

In [ ]:
# Model 1: Simple model, 100 epochs
tf.random.set_seed(42)
model_1 = tf.keras.Sequential([tf.keras.layers.Dense(1)])
model_1.compile(loss=tf.keras.losses.mae, optimizer=tf.keras.optimizers.SGD(), metrics=['mae'])
model_1.fit(tf.expand_dims(X_train, axis=-1), y_train, epochs=100, verbose=0)

# Make predictions and evaluate
y_preds_1 = model_1.predict(X_test, verbose=0)
mae_1 = MeanAbsoluteError()(y_test, y_preds_1.squeeze()).numpy()
mse_1 = MeanSquaredError()(y_test, y_preds_1.squeeze()).numpy()
print(f"Model 1 - MAE: {mae_1:.4f}, MSE: {mse_1:.4f}")

In [ ]:
# Model 2: Two layers, 100 epochs
tf.random.set_seed(42)
model_2 = tf.keras.Sequential([
    tf.keras.layers.Dense(1),
    tf.keras.layers.Dense(1)  # Added second layer
])
model_2.compile(loss=tf.keras.losses.mae, optimizer=tf.keras.optimizers.SGD(), metrics=['mae'])
model_2.fit(tf.expand_dims(X_train, axis=-1), y_train, epochs=100, verbose=0)

# Make predictions and evaluate
y_preds_2 = model_2.predict(X_test, verbose=0)
mae_2 = MeanAbsoluteError()(y_test, y_preds_2.squeeze()).numpy()
mse_2 = MeanSquaredError()(y_test, y_preds_2.squeeze()).numpy()
print(f"Model 2 - MAE: {mae_2:.4f}, MSE: {mse_2:.4f}")

In [ ]:
# Model 3: Two layers, 500 epochs
tf.random.set_seed(42)
model_3 = tf.keras.Sequential([
    tf.keras.layers.Dense(1),
    tf.keras.layers.Dense(1)
])
model_3.compile(loss=tf.keras.losses.mae, optimizer=tf.keras.optimizers.SGD(), metrics=['mae'])
model_3.fit(tf.expand_dims(X_train, axis=-1), y_train, epochs=500, verbose=0)

# Make predictions and evaluate
y_preds_3 = model_3.predict(X_test, verbose=0)
mae_3 = MeanAbsoluteError()(y_test, y_preds_3.squeeze()).numpy()
mse_3 = MeanSquaredError()(y_test, y_preds_3.squeeze()).numpy()
print(f"Model 3 - MAE: {mae_3:.4f}, MSE: {mse_3:.4f}")

## Comparing Results

Let's compare all three models to see which performed best.

In [ ]:
# Create comparison DataFrame
model_results = [
    ["Model_1 (1 layer, 100 epochs)", mae_1, mse_1],
    ["Model_2 (2 layers, 100 epochs)", mae_2, mse_2],
    ["Model_3 (2 layers, 500 epochs)", mae_3, mse_3]
]

all_results = pd.DataFrame(model_results, columns=["Model", "MAE", "MSE"])
print("Model Comparison:")
print(all_results.to_string(index=False))

# Find best model
best_idx = all_results["MAE"].idxmin()
print(f"\nBest model: {all_results.loc[best_idx, 'Model']}")

## Saving a Model

There are two ways to save a model in TensorFlow:
1. **SavedModel format** - default format (`.keras` extension)
2. **HDF5 format** - older format (`.h5` extension)

In [ ]:
# Save the best model (model_2) in Keras format
model_2.save("best_regression_model.keras")
print("Model saved as 'best_regression_model.keras'")

## Loading a Model

Loading a saved model is straightforward:

In [ ]:
# Load the saved model
loaded_model = tf.keras.models.load_model("best_regression_model.keras")
print("Model loaded successfully!")

# Make prediction with loaded model
test_prediction = loaded_model.predict(np.array([[50.0]]), verbose=0)
print(f"\nPrediction for X=50.0: {test_prediction[0][0]:.2f}")
print(f"Expected (50 + 10): 60.0")

## Summary

In this notebook, we covered:

1. **Creating data** for regression problems
2. **Understanding input/output shapes** - fundamental concept for neural networks
3. **Building models** with TensorFlow's Sequential API
4. **Training** models with compile and fit
5. **Evaluating** models with MAE and MSE metrics
6. **Running experiments** to improve models
7. **Saving and loading** trained models

**Key Takeaways:**
- Regression = predicting a number
- Input/output shapes are crucial
- More layers and longer training can help (but not always!)
- Always compare experiments with consistent metrics
- Save your best models for later use